## Set up the original spatial and temporal grid


## Inverse composition implements a declared projected polynomial

The visible native iterate qfunc loads the padded temperature field, composes
the block or its adjoint and uses within/apply for projector phases.
FlagQuantum constructs those controlled predicates and inverse operations.
Host code supplies finite differences, stability condition, polynomial fit,
normalization and readout gauge; synth does not discover a heat solver.

Inspect all four generated phase values while preserving the source's actual
use of only the first three. The Wx signal and X readout convention matters.
Try another admissible initial temperature vector with the same padding and
compare unnormalized projection against the independent matrix. Projection
norm, polynomial response error and heat-equation discretization error are
separate; a normalized field alone omits success probability.


In [ ]:
from flagquantum import synth
from flagquantum.synth import H
from flagquantum.synth import Output
from flagquantum.synth import QBit
from flagquantum.synth import QNum
from flagquantum.synth import adjoint
from flagquantum.synth import allocate
from flagquantum.synth import apply
from flagquantum.synth import inplace_prepare_amplitudes
from flagquantum.synth import phase
from flagquantum.synth import qfunc
from flagquantum.synth import within
from typing import Annotated
import torch

from examples.synthesis.applications.cfd.heat_eq_qsvt.overview.quantum import block


@qfunc
def iterate(
    angle: float,
    data: list[float],
    phases: list[float],
    data_and_deletion: Output[Annotated[QNum, 4]],
    selection: Output[Annotated[QNum, 2]],
    rotation: Output[QBit],
    aux: Output[QBit],
) -> None:
    allocate(data_and_deletion)
    allocate(selection)
    allocate(rotation)
    allocate(aux)
    inplace_prepare_amplitudes(data, 0, data_and_deletion)
    H(aux)
    for step in range(3):
        if step == 1:
            block(angle, data_and_deletion, selection, rotation)
        elif step == 2:
            with adjoint():
                block(angle, data_and_deletion, selection, rotation)
        with within():
            mark: Annotated[QNum, 1] = (
                (selection == 0) & (rotation == 0) & (data_and_deletion < 8)
            )
        with apply():
            phase(aux - 0.5, phases[step])
            phase((1 - 2 * aux) * mark, phases[step])
    H(aux)


def build_preview(
    angle: float, data: torch.Tensor, phases: tuple[float, ...]
) -> synth.Program:
    return synth.synthesize(
        iterate,
        classical_inputs={
            "angle": angle,
            "data": tuple(map(float, data)) + (0.0,) * 8,
            "phases": phases,
        },
        max_qubits=11,
        max_workspace=3,
        memory_limit_bytes=64 * 1024**2,
    )


from IPython.display import Image, display

from examples.synthesis.applications.cfd.heat_eq_qsvt.overview.main import (
    report,
    save_figures,
)
from examples.synthesis.applications.cfd.heat_eq_qsvt.overview.model import (
    DT,
    DX,
    POINTS,
    TIME_STEPS,
)
from examples.synthesis.applications.cfd.heat_eq_qsvt.overview.workflow import (
    run_workflow,
)

assert DT <= DX**2
print("Spatial points and time samples:", POINTS, TIME_STEPS)


## Initialize the sine profile


In [ ]:
result = run_workflow()
figures = save_figures(result)
display(Image(filename=str(figures[0])))


## Apply the original zero boundary conditions


In [ ]:
for figure in figures[1:3]:
    display(Image(filename=str(figure)))


## Construct the heat operator and normalize the input


In [ ]:
print("Initial input norm:", result.model.initial_norm)
display(Image(filename=str(figures[3])))


## Inspect the finite-difference operator


In [ ]:
print(result.model.operator)
display(Image(filename=str(figures[4])))


## Use local native phase generation


In [ ]:
print("Degree-three phase generation and quantum execution run locally.")


## Select the original Wx and X-readout convention


In [ ]:
print("Signal and readout convention: Wx and X.")


## Read the actual selected branch

The local result uses the phase of the first occupied native basis amplitude. The signed field depends on this convention. The source hosted run recorded near-zero real projections.


In [ ]:
print("Native anchor indices:", result.gauge_indices)


## Build the packed shift and rotation block


In [ ]:
print("Block rotation and rescaling:", result.model.rotation, result.model.scaling)


## Synthesize a fresh probe for each basis input


In [ ]:
print("Native probe columns:", result.measured.shape[1])


## Inspect the complete block reconstruction


In [ ]:
print(result.measured.real)


## Verify the eight basis inputs


In [ ]:
assert result.measured.shape == (8, 8)


## Compare every element with the classical operator


In [ ]:
print("Block matrix norm error:", result.matrix_error)
display(Image(filename=str(figures[5])))


## Fit and validate the fresh phase sequence


In [ ]:
print("Polynomial degree:", 3)
print("Fresh phase response error:", result.fit.max_error)
display(Image(filename=str(figures[6])))


## Execute the source projector order


In [ ]:
assert len(result.fit.phases) == 4
print("Angles, of which the circuit uses the first three:", result.fit.phases)


## Use all four observed heat iterations


In [ ]:
print("Measured projection norms:", result.projection_norms)
assert len(result.projection_norms) == 4


## Review the final synthesized program


In [ ]:
report(result)


## Read physical allocation and depth


In [ ]:
print(
    "Native physical width and depth:",
    result.resources.n_qubits,
    result.resources.depth,
)


## Plot the complete heat profiles


In [ ]:
display(Image(filename=str(figures[7])))
